In [129]:
import glob
import PIL.Image as Image
import torchvision.transforms.v2 as v2
import tqdm
import numpy as np

In [130]:
real_train_imgs = glob.glob("cifar/*.png")
fake_train_imgs = glob.glob("uvit/*.png")

In [131]:
trans = v2.Compose([
    v2.RandomVerticalFlip(),
    v2.RandomHorizontalFlip()
])

In [132]:
train_X = []
train_y = []

for i, path in tqdm.tqdm(enumerate(real_train_imgs)):
    im = Image.open(path).resize((32, 32))

    train_X.append(np.array(trans(im)))
    train_y.append(1)

for i, path in tqdm.tqdm(enumerate(fake_train_imgs)):
    im = Image.open(path).resize((32, 32))

    train_X.append(np.array(trans(im)))
    train_y.append(0)
    

5000it [00:04, 1108.82it/s]
5000it [00:04, 1176.70it/s]


In [133]:
train = [(train_X[i], train_y[i]) for i in range(len(train_X))]

In [134]:
from sklearn.utils import shuffle

train = shuffle(train)

In [135]:
train_X = [x[0] for x in train]
train_y = [x[1] for x in train]


In [136]:
train_X = np.array(train_X)
train_y = np.array(train_y)

In [137]:
print(train_X.shape)
print(train_y.shape)

(10000, 32, 32, 3)
(10000,)


In [192]:
import torch
import torch.nn as nn

# class MyModel(nn.Module):
#     def __init__(self):
#         super(MyModel, self).__init__()

#         self.conv1 = nn.Conv2d(3, 32, kernel_size=(3, 3), padding=1, stride=1)
#         self.conv2 = nn.Conv2d(32, 64, kernel_size=(5, 5), padding=3, stride=1)

#         self.flat = nn.Flatten()
#         self.drop = nn.Dropout()
#         self.norm32 = nn.BatchNorm2d(32)
#         self.norm64 = nn.BatchNorm2d(64)

#         self.maxpool_2x2_1 = nn.MaxPool2d((2, 2))
#         self.maxpool_2x2_2 = nn.MaxPool2d((2, 2))

#         self.linear1 = nn.Linear(5184, 256) 
#         self.relu = nn.ReLU()
#         self.linear2 = nn.Linear(256, 2)
#     def forward(self, x):
#         self.out_conv1 = self.conv1(x)
#         self.out_conv1_pool = self.relu(self.maxpool_2x2_1(self.out_conv1))
#         self.out_conv1_pool_norm = self.drop(self.norm32(self.out_conv1_pool))

#         self.out_conv2 = self.conv2(self.out_conv1_pool_norm)
#         self.out_conv2_pool = self.relu(self.maxpool_2x2_2(self.out_conv2))
#         self.out_conv2_pool_norm = self.drop(self.norm64(self.out_conv2_pool))

#         self.x_flat = self.flat(self.out_conv2_pool_norm)

#         self.out1 = self.relu(self.linear1(self.x_flat))
#         self.out2 = self.linear2(self.out1)

#         return self.out2
    

class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 20, 3)
        self.conv3 = nn.Conv2d(20, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv4 = nn.Conv2d(32, 40, 5)
        self.conv5 = nn.Conv2d(40, 64, 3)
        self.conv6 = nn.Conv2d(64, 128, 5)
        self.pool = nn.MaxPool2d(2, 2)
        

        self.fc1 = nn.Linear(128, 70)
        self.fc2 = nn.Linear(70, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = (torch.relu(self.conv1(x)))
        x = (torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = (torch.relu(self.conv4(x)))
        x = (torch.relu(self.conv5(x)))
        x = self.pool(torch.relu(self.conv6(x)))
        # print(x.shape)  # Debug line to see the shape
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x


In [ ]:
skididi_model = MyModel()



In [201]:
opt = torch.optim.Adam(skididi_model.parameters(), lr=1e-4)
loss_fn = nn.BCELoss()

batch_size = 128

In [202]:
train_X_tens = torch.Tensor(train_X).permute(0, 3, 1, 2)
train_y_tens = torch.Tensor(train_y).reshape(-1, 1)

In [203]:
def get_acc(test_X, test_y, model):
    model.eval()
    out = model(test_X).detach().numpy()
    # print(out)
    pred = [1 if x >= 0.5 else 0 for x in out]

    # print(out)

    acc = (np.array(pred).reshape(-1) == test_y.detach().numpy().reshape(-1)).sum() / len(out)
    return acc


def train_(epoch):
    print("Epoch:", epoch)

    loss_curr = 0;

    for i in (pb := tqdm.tqdm(range(0, len(train_X), batch_size))):
        X = train_X_tens[i:i + batch_size]
        y = train_y_tens[i:i + batch_size]
        # print(X.shape)

        output = skididi_model(X)
        loss = loss_fn(output, y)
        loss_curr += loss.item()
        
        pb.set_postfix({"Loss": loss_curr / (i + 1)})

        opt.zero_grad()
        loss.backward()
        opt.step()
    print(get_acc(train_X_tens, train_y_tens, skididi_model))
    
    print("Loss:", loss_curr / len(train))
        

In [204]:
for i in range(100):
    train_(i)

Epoch: 0


100%|██████████| 79/79 [00:09<00:00,  8.59it/s, Loss=0.000505]


0.9914
Loss: 0.0005041266052037826
Epoch: 1


100%|██████████| 79/79 [00:08<00:00,  9.34it/s, Loss=0.000207]


0.9963
Loss: 0.0002065896593107027
Epoch: 2


...

0.9999
Loss: 4.078673936182895e-05
Epoch: 5


100%|██████████| 79/79 [00:09<00:00,  8.67it/s, Loss=2.56e-5]


0.9999
Loss: 2.560576383575608e-05
Epoch: 6


100%|██████████| 79/79 [00:09<00:00,  8.65it/s, Loss=1.72e-5]


KeyboardInterrupt: 

In [205]:
model_code = """  
import torch
import torch.nn as nn
import torch.optim as optim
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3)
        self.conv2 = nn.Conv2d(16, 20, 3)
        self.conv3 = nn.Conv2d(20, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv4 = nn.Conv2d(32, 40, 5)
        self.conv5 = nn.Conv2d(40, 64, 3)
        self.conv6 = nn.Conv2d(64, 128, 5)
        self.pool = nn.MaxPool2d(2, 2)
        

        self.fc1 = nn.Linear(128, 70)
        self.fc2 = nn.Linear(70, 1)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        x = (torch.relu(self.conv1(x)))
        x = (torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = (torch.relu(self.conv4(x)))
        x = (torch.relu(self.conv5(x)))
        x = self.pool(torch.relu(self.conv6(x)))
        # print(x.shape)  # Debug line to see the shape
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

"""
# Write the model code into a file.
with open('submission_model.py', 'w') as f:
    f.write(model_code)
print("submission_model.py file is generated.")
torch.save(skididi_model.state_dict(), 'submission_dic.pth')

submission_model.py file is generated.


In [206]:
print(get_acc(train_X_tens, train_y_tens, skididi_model))

0.9999


In [207]:
import zipfile
import os

# Define the files to be packaged and the compressed file name.
files_to_zip = ['submission_model.py', 'submission_dic.pth']
zip_filename = 'submission.zip'

# Create a zip file
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        # 将Add files to the zip file.
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created successfully!')

submission.zip is created successfully!
